In [1]:
import mwclient ##
import time

site = mwclient.Site("en.wikipedia.org")
page = site.pages["Bitcoin"]

In [2]:
revs = list(page.revisions())

In [3]:
revs[0]

OrderedDict([('revid', 1177224527),
             ('parentid', 1177221816),
             ('minor', ''),
             ('user', 'AnomieBOT'),
             ('timestamp',
              time.struct_time(tm_year=2023, tm_mon=9, tm_mday=26, tm_hour=19, tm_min=50, tm_sec=8, tm_wday=1, tm_yday=269, tm_isdst=-1)),
             ('comment', 'Dating maintenance tags: {{Expand section}}')])

In [4]:
revs = sorted(revs, key=lambda rev: rev["timestamp"])

In [5]:
revs[0] ##revid parentid user keys

OrderedDict([('revid', 275832581),
             ('parentid', 0),
             ('user', 'Pratyeka'),
             ('timestamp',
              time.struct_time(tm_year=2009, tm_mon=3, tm_mday=8, tm_hour=16, tm_min=41, tm_sec=7, tm_wday=6, tm_yday=67, tm_isdst=-1)),
             ('comment', 'creation (stub)')])

In [6]:
revs = sorted(revs, key=lambda rev: rev["timestamp"])

In [7]:
revs[0]

OrderedDict([('revid', 275832581),
             ('parentid', 0),
             ('user', 'Pratyeka'),
             ('timestamp',
              time.struct_time(tm_year=2009, tm_mon=3, tm_mday=8, tm_hour=16, tm_min=41, tm_sec=7, tm_wday=6, tm_yday=67, tm_isdst=-1)),
             ('comment', 'creation (stub)')])

In [8]:
##Using Sentiment Analysis Higging Face 

from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

def find_sentiment(text):
    # Limit the text to 250 characters as some models have input size limitations
    text = text[:250]
    
    # Analyze sentiment
    sent = sentiment_pipeline(text)[0]
    score = sent["score"]
    
    # Check sentiment label and adjust the score if it's negative
    if sent["label"] == "NEGATIVE":
        score *= -1
    
    return score


No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


In [9]:
edits = {}

for rev in revs:        
    date = time.strftime("%Y-%m-%d", rev["timestamp"])
    if date not in edits:
        edits[date] = dict(sentiments=list(), edit_count=0)
    
    edits[date]["edit_count"] += 1
    
    comment = rev.get("comment", "")
    edits[date]["sentiments"].append(find_sentiment(comment))

In [10]:
from statistics import mean

for key in edits:
    if len(edits[key]["sentiments"]) > 0:
        edits[key]["sentiment"] = mean(edits[key]["sentiments"])
        edits[key]["neg_sentiment"] = len([s for s in edits[key]["sentiments"] if s < 0]) / len(edits[key]["sentiments"])
    else:
        edits[key]["sentiment"] = 0
        edits[key]["neg_sentiment"] = 0
    
    del edits[key]["sentiments"]

In [11]:
import pandas as pd

edits_df = pd.DataFrame.from_dict(edits, orient="index")

In [12]:
edits_df

,edit_count,sentiment,neg_sentiment
2009-03-08,4,-0.550525,0.750
2009-08-05,1,0.748121,0.000
2009-08-06,2,0.995746,0.000
2009-08-14,1,0.930021,0.000
2009-10-13,2,-0.227501,0.500
...,...,...,...
2023-09-13,8,-0.231592,0.625
2023-09-14,4,0.007662,0.500
2023-09-21,1,-0.999564,1.000
2023-09-22,1,-0.993434,1.000


In [13]:
edits_df.index = pd.to_datetime(edits_df.index)

In [14]:
from datetime import datetime

dates = pd.date_range(start="2009-03-08",end=datetime.today())

In [15]:
dates

DatetimeIndex(['2009-03-08', '2009-03-09', '2009-03-10', '2009-03-11',
               '2009-03-12', '2009-03-13', '2009-03-14', '2009-03-15',
               '2009-03-16', '2009-03-17',
               ...
               '2023-09-25', '2023-09-26', '2023-09-27', '2023-09-28',
               '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02',
               '2023-10-03', '2023-10-04'],
              dtype='datetime64[ns]', length=5324, freq='D')

In [16]:
edits_df = edits_df.reindex(dates, fill_value=0)

In [17]:
edits_df

,edit_count,sentiment,neg_sentiment
2009-03-08,4,-0.550525,0.75
2009-03-09,0,0.000000,0.00
2009-03-10,0,0.000000,0.00
2009-03-11,0,0.000000,0.00
2009-03-12,0,0.000000,0.00
...,...,...,...
2023-09-30,0,0.000000,0.00
2023-10-01,0,0.000000,0.00
2023-10-02,0,0.000000,0.00
2023-10-03,0,0.000000,0.00


In [18]:
rolling_edits = edits_df.rolling(30, min_periods=30).mean()


In [19]:
rolling_edits = rolling_edits.dropna()

In [20]:
rolling_edits

,edit_count,sentiment,neg_sentiment
2009-04-06,0.133333,-0.018351,0.025000
2009-04-07,0.000000,0.000000,0.000000
2009-04-08,0.000000,0.000000,0.000000
2009-04-09,0.000000,0.000000,0.000000
2009-04-10,0.000000,0.000000,0.000000
...,...,...,...
2023-09-30,0.766667,-0.121405,0.193056
2023-10-01,0.766667,-0.121405,0.193056
2023-10-02,0.766667,-0.121405,0.193056
2023-10-03,0.766667,-0.121405,0.193056


In [21]:
rolling_edits.to_csv("wikipedia_edits.csv")

In [57]:
######